In [ ]:
%cd ../..

In [ ]:
from pathlib import Path

import polars as pl

# Read raw data

In [ ]:
paths_1 = Path("data/raw/biowaste/Apr_1").glob("*.csv")
paths_2 = Path("data/raw/biowaste").glob("*.xlsx")

paths = list(paths_1) + list(paths_2)

cols = ['pcs', 'date', 'meal', 'waste', 'restaurant']

df_list = []

for path in paths:
    df = (
        pl.read_csv(path, separator=";")
        if path.suffix == ".csv"
        else pl.read_excel(path)
    )

    df.columns = cols
    df = df.with_columns(
        pl.lit(path.stem).alias('src'),
        pl.col('waste').cast(pl.Float32),
    )

    df_list.append(df)

waste_raw = pl.concat(df_list)
waste_raw.head()

In [ ]:
path = "data/processed/dim_restaurants.xlsx"
dim_restaurants = pl.read_excel(path)

dim_restaurants.head()

# Process

In [ ]:
waste = (
    waste_raw
    .with_columns(
        pl.coalesce(
            pl.col('date').str.to_date("%d.%m.%Y", strict=False),
            pl.col('date').str.to_date("%Y-%m-%d", strict=False, exact=False)
        ).alias('date'),
    )

    .join(dim_restaurants, on='restaurant', how='left')
    .drop('restaurant')
    .rename({'restaurant_id': 'restaurant'})

    .group_by('date', 'restaurant')
    .agg(
        pl.concat_list('waste').flatten().unique(),
        pl.concat_list('src').flatten().unique()
    )
    .with_columns(
        pl.col('waste').list.max()
    )
)

# Save

In [ ]:
path = "data/processed/waste.parquet"

waste.write_parquet(path)